In [0]:
dbutils.widgets.text("catalog","rideflow")
catalog=dbutils.widgets.get("catalog")

landing=f"/Volumes/{catalog}/raw/landing_zone"
chk_root=f"/Volumes/{catalog}/raw/checkpoints"

In [0]:
from pyspark.sql import functions as F

source=f"{landing}/trips"

df=(
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format","json")
    .option("cloudFiles.schemalocation",f"{chk_root}/trips_raw/schema")
    .option("cloudFiles.inferColumnTypes","true")
    .load(source)

)

In [0]:
df.select("_metadata").printSchema()

In [0]:
df=(
    df.withColumn("_source_file" ,F.col("_metadata.file_path"))
    .withColumn("_file_mod_time" ,F.col("_metadata.file_modification_time"))
    .withColumn("_ingested_at",F.current_timestamp()
))

In [0]:
q=(
    df.writeStream
    .option("checkpointLocation",f"{chk_root}/trips_raw/checkpoints")
    .trigger(availableNow=True)
    .toTable(f"{catalog}.bronze.trips_raw")
)

q.awaitTermination()

In [0]:
%sql 
select count(*), count(distinct _source_file) from rideflow.bronze.trips_raw

In [0]:
def ingest(source_subdir, file_format, target_table):
    source = f"{landing}/{source_subdir}"
    chk    = f"{chk_root}/{target_table}"

    reader = (spark.readStream
                .format("cloudFiles")
                .option("cloudFiles.format", file_format)
                .option("cloudFiles.schemaLocation", f"{chk}/schema")
                .option("cloudFiles.inferColumnTypes", "true"))

    if file_format == "csv":                        
        reader = reader.option("header", "true")   

    df = (reader.load(source)
            .withColumn("_source_file",   F.col("_metadata.file_path"))
            .withColumn("_file_mod_time", F.col("_metadata.file_modification_time"))
            .withColumn("_ingested_at",   F.current_timestamp()))

    q = (df.writeStream
           .option("checkpointLocation", f"{chk}/checkpoints")
           .trigger(availableNow=True)
           .toTable(f"{catalog}.bronze.{target_table}"))
    q.awaitTermination()
    print(f"{target_table}: done")                 

In [0]:
ingest("trips",       "json", "trips_raw")
ingest("payments",    "csv",  "payments_raw")
ingest("cdc/drivers", "json", "drivers_cdc_raw")

In [0]:
%sql 
select count(*), count(distinct _source_file) from rideflow.bronze.drivers_cdc_raw;

In [0]:
%sql 
 select count(*), count(distinct _source_file) from rideflow.bronze.trips_raw;


In [0]:
%sql
select count(*), count(distinct _source_file) from rideflow.bronze.payments_raw;

In [0]:
for name in ["cities", "riders", "vehicles"]:
    (spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(f"{landing}/master/{name}.csv")          
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .withColumn("_ingested_at", F.current_timestamp())
        .write
        .mode("overwrite")
        .saveAsTable(f"{catalog}.bronze.{name}"))     
    print(f"{name}: done")

In [0]:
%sql
select count(*), count(distinct _source_file) from rideflow.bronze.cities;

In [0]:
%sql
select count(*), count(distinct _source_file) from rideflow.bronze.riders;

In [0]:
%sql
select count(*), count(distinct _source_file) from rideflow.bronze.vehicles;